In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU name: Tesla T4


In [2]:
import os
import numpy as np
import mne

mne.set_log_level('WARNING')

epoched_dir = '/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels'
subjects = [f"{i:02d}" for i in range(1, 10)]
sessions = ['T', 'E']

all_epochs = {}
all_labels = {}

for subj in subjects:
    for sess in sessions:
        key = f"A{subj}{sess}"
        epo_path = os.path.join(epoched_dir, f"{key}_epo.fif")
        label_path = os.path.join(epoched_dir, f"{key}_labels.npy")
        if os.path.exists(epo_path) and os.path.exists(label_path):
            all_epochs[key] = mne.read_epochs(epo_path, preload=True, verbose=False)
            all_labels[key] = np.load(label_path)

print(f"Loaded {len(all_epochs)}/18 subject/session epoch files")

Loaded 18/18 subject/session epoch files


In [3]:
!pip install braindecode -q

import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import Dataset, DataLoader
from braindecode.models import ShallowFBCSPNet
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from scipy import stats
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42
print("Using device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.9/626.9 kB 15.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 12.4 MB/s eta 0:00:00
Using device: cuda


In [5]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def train_model(model, train_loader, val_loader, n_epochs=100, lr=0.001, patience=15, weight_decay=0.0, verbose=True):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_correct += (outputs.argmax(1) == y_batch).sum().item()
            train_total += y_batch.size(0)
        train_loss /= train_total
        train_acc = train_correct / train_total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()
                val_total += y_batch.size(0)
        val_loss /= val_total
        val_acc = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if verbose and (epoch % 10 == 0 or epoch == n_epochs - 1):
            print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} "
                  f"| Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                break

    model.load_state_dict(best_model_state)
    return model, history

In [6]:
def crop_array(X, y, crop_len=500, stride=50):
    """
    X: (n_trials, n_channels, n_timepoints), y: (n_trials,)
    Slides a window across each trial, generating multiple crops per trial.
    """
    cropped_X, cropped_y, trial_ids = [], [], []
    for i in range(len(X)):
        start = 0
        while start + crop_len <= X.shape[2]:
            cropped_X.append(X[i][:, start:start+crop_len])
            cropped_y.append(y[i])
            trial_ids.append(i)   # track which original trial each crop came from (needed for majority-vote later)
            start += stride
    return np.array(cropped_X), np.array(cropped_y), np.array(trial_ids)

print("All setup functions defined.")

All setup functions defined.


In [7]:
def run_subject_pipeline_cropped(subject_id, crop_len=500, stride=50,
                                    n_epochs=100, lr=0.001, patience=15,
                                    weight_decay=0.01, drop_prob=0.5,
                                    batch_size=64, val_frac=0.2, seed=42):
    """
    Full pipeline for one subject using cropped/augmented training:
    1. Trial-level train/val split (leakage-safe)
    2. Crop each split independently
    3. Train ShallowFBCSPNet on cropped train data
    4. Evaluate on E-session: crop each test trial, predict per-crop, majority-vote per trial
    """
    X_train_full = all_epochs[f"A{subject_id}T"].get_data()
    y_train_full = all_labels[f"A{subject_id}T"] - 1
    X_test_full = all_epochs[f"A{subject_id}E"].get_data()
    y_test_full = all_labels[f"A{subject_id}E"] - 1

    # --- Step A: trial-level train/val split (BEFORE any cropping) ---
    n_trials = len(X_train_full)
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n_trials)
    n_val = int(val_frac * n_trials)
    val_idx, train_idx = indices[:n_val], indices[n_val:]

    X_tr_trials, y_tr_trials = X_train_full[train_idx], y_train_full[train_idx]
    X_val_trials, y_val_trials = X_train_full[val_idx], y_train_full[val_idx]

    # --- Step B: crop each split independently ---
    X_tr_c, y_tr_c, _ = crop_array(X_tr_trials, y_tr_trials, crop_len, stride)
    X_val_c, y_val_c, _ = crop_array(X_val_trials, y_val_trials, crop_len, stride)

    # --- Step C: normalize (fit on cropped train only) ---
    mean = X_tr_c.mean(axis=(0, 2), keepdims=True)
    std = X_tr_c.std(axis=(0, 2), keepdims=True)
    X_tr_norm = (X_tr_c - mean) / (std + 1e-8)
    X_val_norm = (X_val_c - mean) / (std + 1e-8)

    train_loader = DataLoader(EEGDataset(X_tr_norm, y_tr_c), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(EEGDataset(X_val_norm, y_val_c), batch_size=batch_size, shuffle=False)

    # --- Step D: train model ---
    model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=crop_len, drop_prob=drop_prob)
    trained, hist = train_model(model, train_loader, val_loader, n_epochs=n_epochs,
                                  lr=lr, patience=patience, weight_decay=weight_decay, verbose=False)

    # --- Step E: evaluate on E-session using crop + majority-vote ---
    X_test_c, y_test_c, test_trial_ids = crop_array(X_test_full, y_test_full, crop_len, stride)
    X_test_norm = (X_test_c - mean) / (std + 1e-8)   # use TRAIN mean/std, not test's own

    test_loader = DataLoader(EEGDataset(X_test_norm, y_test_c), batch_size=batch_size, shuffle=False)

    trained.eval()
    crop_preds = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(device)
            out = trained(Xb)
            crop_preds.extend(out.argmax(1).cpu().numpy())
    crop_preds = np.array(crop_preds)

    # majority vote per original trial
    n_test_trials = len(X_test_full)
    final_preds = []
    for trial_i in range(n_test_trials):
        mask = test_trial_ids == trial_i
        votes = crop_preds[mask]
        majority = np.bincount(votes).argmax()
        final_preds.append(majority)
    final_preds = np.array(final_preds)

    test_acc = accuracy_score(y_test_full, final_preds)
    test_kappa = cohen_kappa_score(y_test_full, final_preds)
    best_val_acc = max(hist['val_acc'])

    print(f"Subject A{subject_id}: best_val_acc={best_val_acc:.4f} | test_acc={test_acc:.4f} | test_kappa={test_kappa:.4f}")

    return {
        'subject': f"A{subject_id}",
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'test_kappa': test_kappa,
        'n_train_crops': len(X_tr_c),
        'n_test_trials': n_test_trials,
    }


# Run for all 9 subjects
results_cropped = []
for subj in subjects:
    result = run_subject_pipeline_cropped(subj)
    results_cropped.append(result)

results_cropped_df = pd.DataFrame(results_cropped)
print("\n=== Summary (Cropped Training + Majority Vote) ===")
print(results_cropped_df)
print(f"\nMean test accuracy: {results_cropped_df['test_acc'].mean():.4f}")
print(f"Mean test kappa: {results_cropped_df['test_kappa'].mean():.4f}")

Subject A01: best_val_acc=0.7862 | test_acc=0.8043 | test_kappa=0.7388
Subject A02: best_val_acc=0.4747 | test_acc=0.5230 | test_kappa=0.3644
Subject A03: best_val_acc=0.7273 | test_acc=0.7656 | test_kappa=0.6871
Subject A04: best_val_acc=0.5892 | test_acc=0.5175 | test_kappa=0.3568
Subject A05: best_val_acc=0.6608 | test_acc=0.4964 | test_kappa=0.3306
Subject A06: best_val_acc=0.5328 | test_acc=0.4558 | test_kappa=0.2738
Subject A07: best_val_acc=0.8333 | test_acc=0.7148 | test_kappa=0.6214
Subject A08: best_val_acc=0.8199 | test_acc=0.6863 | test_kappa=0.5817
Subject A09: best_val_acc=0.7157 | test_acc=0.6515 | test_kappa=0.5358

=== Summary (Cropped Training + Majority Vote) ===
  subject  best_val_acc  test_acc  test_kappa  n_train_crops  n_test_trials
0     A01      0.786195  0.804270    0.738834           2409            281
1     A02      0.474747  0.522968    0.364426           2376            283
2     A03      0.727273  0.765568    0.687145           2376            273
3    

In [8]:
# Step 1: Build a combined cross-subject pretraining dataset (all 9 subjects' T-session, cropped)

crop_len = 500
stride = 50

all_pretrain_X = []
all_pretrain_y = []

for subj in subjects:
    X = all_epochs[f"A{subj}T"].get_data()
    y = all_labels[f"A{subj}T"] - 1
    X_c, y_c, _ = crop_array(X, y, crop_len, stride)
    all_pretrain_X.append(X_c)
    all_pretrain_y.append(y_c)

X_pretrain = np.concatenate(all_pretrain_X, axis=0)
y_pretrain = np.concatenate(all_pretrain_y, axis=0)

print("Combined pretraining data shape:", X_pretrain.shape)
print("Label distribution:", dict(zip(*np.unique(y_pretrain, return_counts=True))))

Combined pretraining data shape: (25608, 22, 500)
Label distribution: {np.int64(0): np.int64(6292), np.int64(1): np.int64(6501), np.int64(2): np.int64(6314), np.int64(3): np.int64(6501)}


In [9]:
# Step 2: Pretrain a single ShallowFBCSPNet on the combined cross-subject data

# Split combined data into pretrain-train/val (just for early stopping during pretraining)
n_total = len(X_pretrain)
rng = np.random.RandomState(RANDOM_SEED)
idx = rng.permutation(n_total)
n_val = int(0.1 * n_total)   # small val split, just to monitor pretraining convergence
val_idx, train_idx = idx[:n_val], idx[n_val:]

X_pre_tr, y_pre_tr = X_pretrain[train_idx], y_pretrain[train_idx]
X_pre_val, y_pre_val = X_pretrain[val_idx], y_pretrain[val_idx]

# Normalize (fit on pretrain-train only) — save these stats, will reuse consistent scaling later
pretrain_mean = X_pre_tr.mean(axis=(0, 2), keepdims=True)
pretrain_std = X_pre_tr.std(axis=(0, 2), keepdims=True)

X_pre_tr_norm = (X_pre_tr - pretrain_mean) / (pretrain_std + 1e-8)
X_pre_val_norm = (X_pre_val - pretrain_mean) / (pretrain_std + 1e-8)

pretrain_loader = DataLoader(EEGDataset(X_pre_tr_norm, y_pre_tr), batch_size=128, shuffle=True)
pretrain_val_loader = DataLoader(EEGDataset(X_pre_val_norm, y_pre_val), batch_size=128, shuffle=False)

print(f"Pretraining on {len(X_pre_tr)} samples, validating on {len(X_pre_val)}")

pretrained_model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=crop_len, drop_prob=0.5)

pretrained_model, pretrain_hist = train_model(
    pretrained_model,
    pretrain_loader,
    pretrain_val_loader,
    n_epochs=100,
    lr=0.001,
    patience=15,
    weight_decay=0.01,
    verbose=True
)

print("\nPretraining complete.")

Pretraining on 23048 samples, validating on 2560
Epoch 1/100 | Train Loss: 1.5122 Acc: 0.3209 | Val Loss: 1.2669 Acc: 0.4234
Epoch 11/100 | Train Loss: 1.0094 Acc: 0.5829 | Val Loss: 0.9912 Acc: 0.5723
Epoch 21/100 | Train Loss: 0.9409 Acc: 0.6162 | Val Loss: 0.8644 Acc: 0.6352
Epoch 31/100 | Train Loss: 0.9014 Acc: 0.6358 | Val Loss: 0.8856 Acc: 0.6465
Epoch 41/100 | Train Loss: 0.8665 Acc: 0.6543 | Val Loss: 0.8067 Acc: 0.6758
Epoch 51/100 | Train Loss: 0.8625 Acc: 0.6530 | Val Loss: 0.8508 Acc: 0.6465
Epoch 61/100 | Train Loss: 0.8264 Acc: 0.6688 | Val Loss: 0.7629 Acc: 0.6930
Early stopping at epoch 61 (no improvement for 15 epochs)

Pretraining complete.


In [11]:
import copy as copy_module

def finetune_subject(subject_id, pretrained_state_dict, crop_len=500, stride=50,
                       n_epochs=50, lr=0.0001, patience=15,   # lower LR for fine-tuning
                       weight_decay=0.01, drop_prob=0.5, batch_size=32, val_frac=0.2, seed=42):
    """
    Loads the pretrained model, fine-tunes on one subject's own T-session data (cropped),
    evaluates on that subject's E-session using crop + majority-vote.
    """
    X_train_full = all_epochs[f"A{subject_id}T"].get_data()
    y_train_full = all_labels[f"A{subject_id}T"] - 1
    X_test_full = all_epochs[f"A{subject_id}E"].get_data()
    y_test_full = all_labels[f"A{subject_id}E"] - 1

    # Trial-level split, then crop
    n_trials = len(X_train_full)
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n_trials)
    n_val = int(val_frac * n_trials)
    val_idx, train_idx = indices[:n_val], indices[n_val:]

    X_tr_trials, y_tr_trials = X_train_full[train_idx], y_train_full[train_idx]
    X_val_trials, y_val_trials = X_train_full[val_idx], y_train_full[val_idx]

    X_tr_c, y_tr_c, _ = crop_array(X_tr_trials, y_tr_trials, crop_len, stride)
    X_val_c, y_val_c, _ = crop_array(X_val_trials, y_val_trials, crop_len, stride)

    # IMPORTANT: use the SAME normalization stats from pretraining (consistent scaling across subjects)
    X_tr_norm = (X_tr_c - pretrain_mean) / (pretrain_std + 1e-8)
    X_val_norm = (X_val_c - pretrain_mean) / (pretrain_std + 1e-8)

    train_loader = DataLoader(EEGDataset(X_tr_norm, y_tr_c), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(EEGDataset(X_val_norm, y_val_c), batch_size=batch_size, shuffle=False)

    # Load a FRESH copy of the pretrained weights (so subjects don't interfere with each other)
    model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=crop_len, drop_prob=drop_prob)
    model.load_state_dict(copy_module.deepcopy(pretrained_state_dict))

    finetuned, hist = train_model(model, train_loader, val_loader, n_epochs=n_epochs,
                                    lr=lr, patience=patience, weight_decay=weight_decay, verbose=False)

    # Evaluate on E-session: crop + majority vote
    X_test_c, y_test_c, test_trial_ids = crop_array(X_test_full, y_test_full, crop_len, stride)
    X_test_norm = (X_test_c - pretrain_mean) / (pretrain_std + 1e-8)

    test_loader = DataLoader(EEGDataset(X_test_norm, y_test_c), batch_size=batch_size, shuffle=False)

    finetuned.eval()
    crop_preds = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(device)
            out = finetuned(Xb)
            crop_preds.extend(out.argmax(1).cpu().numpy())
    crop_preds = np.array(crop_preds)

    n_test_trials = len(X_test_full)
    final_preds = []
    for trial_i in range(n_test_trials):
        mask = test_trial_ids == trial_i
        votes = crop_preds[mask]
        majority = np.bincount(votes).argmax()
        final_preds.append(majority)
    final_preds = np.array(final_preds)

    test_acc = accuracy_score(y_test_full, final_preds)
    test_kappa = cohen_kappa_score(y_test_full, final_preds)
    best_val_acc = max(hist['val_acc'])

    print(f"Subject A{subject_id}: best_val_acc={best_val_acc:.4f} | test_acc={test_acc:.4f} | test_kappa={test_kappa:.4f}")

    return {'subject': f"A{subject_id}", 'best_val_acc': best_val_acc,
            'test_acc': test_acc, 'test_kappa': test_kappa}


# Get pretrained weights
pretrained_state = pretrained_model.state_dict()

# Run fine-tuning for all 9 subjects
results_finetuned = []
for subj in subjects:
    result = finetune_subject(subj, pretrained_state)
    results_finetuned.append(result)

results_finetuned_df = pd.DataFrame(results_finetuned)
print("\n=== Summary (Cross-Subject Pretrain + Per-Subject Fine-tune) ===")
print(results_finetuned_df)
print(f"\nMean test accuracy: {results_finetuned_df['test_acc'].mean():.4f}")
print(f"Mean test kappa: {results_finetuned_df['test_kappa'].mean():.4f}")

Subject A01: best_val_acc=0.8418 | test_acc=0.8648 | test_kappa=0.8197
Subject A02: best_val_acc=0.6162 | test_acc=0.4947 | test_kappa=0.3270
Subject A03: best_val_acc=0.8535 | test_acc=0.8828 | test_kappa=0.8436
Subject A04: best_val_acc=0.7150 | test_acc=0.5702 | test_kappa=0.4274
Subject A05: best_val_acc=0.7990 | test_acc=0.5290 | test_kappa=0.3750
Subject A06: best_val_acc=0.6681 | test_acc=0.5488 | test_kappa=0.3979
Subject A07: best_val_acc=0.8603 | test_acc=0.8773 | test_kappa=0.8363
Subject A08: best_val_acc=0.8776 | test_acc=0.7565 | test_kappa=0.6756
Subject A09: best_val_acc=0.7969 | test_acc=0.6932 | test_kappa=0.5908

=== Summary (Cross-Subject Pretrain + Per-Subject Fine-tune) ===
  subject  best_val_acc  test_acc  test_kappa
0     A01      0.841751  0.864769    0.819680
1     A02      0.616162  0.494700    0.326964
2     A03      0.853535  0.882784    0.843648
3     A04      0.715035  0.570175    0.427400
4     A05      0.798951  0.528986    0.374978
5     A06      0.66